# Testing RAG model

In [ ]:
# !pip install -q -U transformers
# !pip install -q sentence-transformers
# !pip install tqdm lancedb gptqmodel


## Connection to PoliMilionaire

### Imports and Installs

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from google.colab import userdata, drive
from huggingface_hub import login
import re
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import Callable
import os
import pandas as pd
from transformers import AutoModelForSeq2SeqLM,AutoModelForCausalLM, AutoTokenizer, pipeline
import sys
import time
from typing import Callable
from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig


In [ ]:
# import phase
# !pip install tqdm lancedb gptqmodel #hnswlib faiss-cpu rank_bm25 beir ir_measures
# !pip install -U bitsandbytes>=0.46.1

# from rank_bm25 import BM25Okapi
# import faiss
#import hnswlib
from sentence_transformers import util
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
import tqdm
from datasets import load_dataset, load_from_disk
import lancedb

### Connections

#### Google Drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Hugging Face

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


#### Game APIs

In [ ]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Repository already present, update...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 35.20 KiB | 563.00 KiB/s, done.
From https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi
   a28204a..5a51a75  rag        -> origin/rag
Already up to date.
Logged in as: GliEmbeddingRuspanti (role: student)


### Model class

In [ ]:
class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"


### The Game

In [ ]:
def play_game(game, model, sys_prompt):
  log = []
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      options = {f"{opt.id}": opt.text for opt in question.options}

      t0 = time.time()
      answer_summary, answer_input = model.answer(question.text, options, sys_prompt)
      inference_time = time.time() - t0
      print(f"Model answer: {answer_input}")
      answer_id = int(answer_input)

      choosen_answer = question.options[answer_id]

      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

      # Log the outcome
      entry = {
          'level'           : game.current_level,
          'question'        : question.text,
          'options'         : question.options,
          'chosen_option'   : choosen_answer.text,
          'correct'         : result.correct,
          'timed_out'       : result.timed_out,
          'inference_time'  : round(inference_time, 2),
          'answer_summary'  : answer_summary,
      }
      log.append(entry)

  summary = {
        'model'           : model.name,
        'final_level'     : game.current_level,
        'earned_amount'   : game.earned_amount,
        'num_questions'   : len(log),
        'num_correct'     : sum(1 for e in log if e['correct']),
        'num_timed_out'   : sum(1 for e in log if e['timed_out']),
        'avg_inference_s' : round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'             : log,
    }

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

  return summary

### RAG model class

In [ ]:
def load_rag_model():
  bi_enc = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.float16})
  reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=1024)
  model_id = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
  tokenizer = AutoTokenizer.from_pretrained(model_id)
  model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
  )

  VECTOR_DB_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local/"
  table_name = "wiki_rag_collection"
  if not os.path.exists(VECTOR_DB_PATH):
    print("ERRORE CRITICO: La cartella base non esiste per Colab!")
  else:
      contenuto = os.listdir(VECTOR_DB_PATH)
      print(f"Cosa c'è fisicamente dentro '{VECTOR_DB_PATH}':")
      print(contenuto)

      if f"{table_name}.lance" in contenuto:
          print(f"\n✅ PERFETTO! La tabella fisica '{table_name}.lance' C'È.")
      else:
          print(f"\n❌ ERRORE! La tabella fisica '{table_name}.lance' MANCA in questa cartella.")
          print("Probabilmente il path è sbagliato o la cartella è nidificata più a fondo.")

  print("\n--- TEST LANCEDB ---")
  db = lancedb.connect(VECTOR_DB_PATH)

  # TRUCCO: Usa table_names() invece di list_tables()
  tabelle_presenti = db.table_names()
  print(f"Tabelle viste da LanceDB: {tabelle_presenti}")

  if table_name in tabelle_presenti:
      collection = db.open_table(table_name)
      print(f"🎉 SUCCESSO! Tabella '{table_name}' aperta.")
  else:
      print(f"❌ FALLIMENTO. LanceDB non vede la tabella.")

  # Assuming DS_PATH (directory for Arrow cache) and PARQUET_PATH are defined earlier
  DS_PATH="/content/drive/MyDrive/Progetto-NLP/Branch-rag/"
  PARQUET_PATH = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection_ita.parquet'

  # We save Hugging Face datasets as a directory structure, not a single file
  ds_arrow_dir = os.path.join(DS_PATH, "ds_embedding_collection_ita")

  ds = None

  # Attempt to load from native Hugging Face Disk Cache (Super Fast Arrow Format)
  if os.path.exists(ds_arrow_dir):
      print("Attempting to load dataset from native disk cache...")
      try:
          ds = load_from_disk(ds_arrow_dir)
          _ = len(ds)  # Quick verification
          print("Dataset loaded successfully from disk cache.")
      except Exception as e:
          print(f"Failed to load dataset from cache ({e}). Attempting to load from raw Parquet instead.")
          ds = None

  # Fallback: If cache doesn't exist or is corrupted, load from Parquet
  if ds is None:
      if os.path.exists(PARQUET_PATH):
          print("Loading dataset from Parquet...")
          ds = load_dataset("parquet", data_files=PARQUET_PATH, split="train")
          print("Dataset loaded successfully from Parquet.")

          # Save it natively to disk for blazing fast future loading
          print("Caching dataset to disk for future use...")
          ds.save_to_disk(ds_arrow_dir)
          print("Dataset cached successfully.")
      else:
          print(f"Error: Neither cache directory ({ds_arrow_dir}) nor Parquet file ({PARQUET_PATH}) found.")
          raise FileNotFoundError(f"Cannot load dataset. Parquet file not found at {PARQUET_PATH}")

  return bi_enc, reranker, model, tokenizer, collection, ds



In [ ]:
def rag_sota(self, query, options_text, top_k=2):
    start_time = time.time()
    print("\nInizio retrieval...")

    # A. RETRIEVAL (LanceDB)
    query_vector = self.bi_enc.encode([query]).tolist()[0]
    risultati = self.collection.search(query_vector).limit(20).to_pandas()

    # B. PREPARAZIONE DOCUMENTI
    retrieved_docs = []
    for _, row in risultati.iterrows():
        doc_id = int(row['id'])
        retrieved_docs.append(self.ds[doc_id]['content'])

    # C. RERANKING
    couples = [[query, doc] for doc in retrieved_docs]
    scores = self.reranker.predict(couples)

    docs_with_score = list(zip(scores, retrieved_docs))
    docs_with_score.sort(key=lambda x: x[0], reverse=True)

    # D. TOP K DOCS (con TRUNCATION DI SICUREZZA)
    top_docs = [doc for score, doc in docs_with_score[:top_k]]
    docs_context = "\n\n---\n\n".join(top_docs)

    if len(docs_context) > 12000:
        docs_context = docs_context[:12000] + "\n... [TRONCATO PER SICUREZZA]"
        print("TRONCATO")

    print("Retrieval finito.")
    end_time_retrivial = time.time()

    # E. PULIZIA RAM
    del risultati
    del retrieved_docs
    del couples
    #gc.collect()
    torch.cuda.empty_cache()

    # F. PROMPTING
    system_prompt = "Sei un risolutore di quiz. Leggi il contesto e restituisci ESCLUSIVAMENTE il numero dell'opzione corretta. Non scrivere altro."

    user_prompt = f"""<contesto>
      {docs_context}
      </contesto>

      Domanda: {query}
      Opzioni: {options_text}

      Istruzione: Il contesto è in italiano, le opzioni in inglese. Analizza il contesto e scrivi SOLO l'ID dell'opzione corretta.
      Risposta:"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt_testo = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # MODIFICA 1: Chiamiamo la variabile 'inputs' e aggiungiamo return_dict=True
    inputs = self.tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True # <-- FONDAMENTALE
    ).to(self.model.device)

    # G. INFERENZA NATIVA ULTRA-VELOCE
    outputs = self.model.generate(
        **inputs,              # <-- MODIFICA 2: Spacchettiamo il dizionario con i due asterischi!
        max_new_tokens=10,
        do_sample=False,
        use_cache=True,
        pad_token_id=self.tokenizer.eos_token_id
    )

    # MODIFICA 3: Dobbiamo prendere la lunghezza da inputs['input_ids']
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    predicted_answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    end_time = time.time()
    tempo_esecuzione = end_time - start_time
    tempo_retrivial = end_time_retrivial - start_time
    print(f"Tempo retrivial: {tempo_retrivial:.2f} secondi")
    print(f"Tempo di esecuzione totale: {tempo_esecuzione:.2f} secondi")

    return prompt_testo, predicted_answer, top_docs

In [ ]:
class RAGModel(Model):
  def __init__(self, name: str):  #, answer_fn: Callable):
    # super().__init__(name, answer_fn)
    self.name=name
    self.bi_enc, self.reranker, self.model, self.tokenizer, self.collection, self.ds = load_rag_model()
    # self.bi_enc = self.reranker = self.model = self.tokenizer = self.collection = self.ds = None

  def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
    _, answer, _ = rag_sota(self, question, options_text=options)
    return _, answer

## Instantiate model

In [ ]:
rag_model = RAGModel("RAG")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

INFO  Kernel: Auto-selection: adding candidate `AwqExllamaV2Linear`            


INFO  Kernel: selected -> `AwqExllamaV2Linear`.                                


[transformers] Current model requires 3399352832 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/739 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.38 GiB is allocated by PyTorch, and 24.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
models = {
    "RAG": {"model": rag_model, "system_prompt": ""}}

## Play and print results

In [ ]:
results = {}

for model_name, config in models.items():
    print(f"\n########## MODEL: {model_name} ##########")

    model = config["model"]
    system_prompt = config["system_prompt"]

    model_results = []

    for comp_id in [0, 1, 2, 3]:
        print(f"\n--- Competition {comp_id} ---")

        game = client.game.start(competition_id=comp_id)

        summary = play_game(game, model, system_prompt)

        model_results.append(summary)

    results[model_name] = model_results

In [ ]:
def print_results(results):
    for model_name, competitions in results.items():

        print("\n" + "=" * 80)
        print(f"MODELLO: {model_name}")
        print("=" * 80)

        for i, summary in enumerate(competitions):

            print(f"\n🏁 Competition {i}")
            print("-" * 60)

            print(f"Model name        : {summary['model']}")
            print(f"Final level       : {summary['final_level']}")
            print(f"Earned amount     : €{summary['earned_amount']}")
            print(f"Questions         : {summary['num_questions']}")
            print(f"Correct answers   : {summary['num_correct']}")
            print(f"Timed out         : {summary['num_timed_out']}")
            print(f"Avg inference     : {summary['avg_inference_s']} s")

            accuracy = (
                summary['num_correct'] / summary['num_questions'] * 100
                if summary['num_questions'] > 0 else 0
            )

            print(f"Accuracy          : {accuracy:.1f}%")

            print("\n📋 Question Log")
            print("-" * 60)

            confidence_array = []

            for q_idx, entry in enumerate(summary['log'], start=1):

                status = "✅" if entry['correct'] else "❌"

                if entry.get('timed_out'):
                    status = "⏰"

                # ─────────────────────────────────────────────
                # CONFIDENCE EXTRACTION (robust fallback chain)
                # ─────────────────────────────────────────────
                answer_summary = entry.get("answer_summary", {})

                if isinstance(answer_summary, dict):
                    if "normalized_margin" in answer_summary:
                        conf = answer_summary["normalized_margin"]

                    elif "confidence" in answer_summary:
                        conf = answer_summary["confidence"]

                    else:
                        conf = None
                else:
                    conf = None

                confidence_array.append(conf)

                print(
                    f"{q_idx:02d}. "
                    f"{status} "
                    f"Time: {entry['inference_time']:.2f}s "
                    f"Conf: {conf if conf is not None else 'N/A'}"
                )

            # ─────────────────────────────────────────────
            # PRINT SUMMARY CONFIDENCE ARRAY
            # ─────────────────────────────────────────────
            print("\n📊 Confidence Array:")
            if any(c is not None for c in confidence_array):
                print(confidence_array)
            else:
                print("Confidence not available")

        print("\n")

# final print
print_results(results)